# nb9.2 — Anticipated-Questions Matrix Generator

*Week 9, Chapter 9.2. Companion notebook.*

The most common way a good empirical talk falls apart at the conference is not because the analysis is
wrong. It is because the speaker has not *enumerated* the questions a careful audience will ask, and so
the first hard one ("did you check for differential pre-trends across cohorts?") lands like a punch in
the dark. A senior referee who has read fifty DiD papers does not have to be brilliant to derail your
talk; they only have to know the small set of *identification threats* every paper in your design class
faces, and ask about the one you forgot to mention.

Priya — who is presenting a difference-in-differences study of a state-level climate-disclosure rule
and its effect on insurer reserves — has been on the receiving end of this twice already. Last spring,
she walked into a faculty seminar with a beautiful event-study plot and got asked, in order, about (i)
the no-anticipation assumption, (ii) two-way-fixed-effects forbidden-comparisons under staggered
adoption, and (iii) whether the SUTVA-violating spillovers across state lines could explain her result.
She knew the answers to two of the three. The third — the one she had not anticipated — ended the
conversation.

The fix is not to be smarter on the day. It is to *generate the matrix of anticipated questions before
the talk*, decide which ones get a backup slide and which get a one-sentence prepared answer, and walk
in with a written defense for each. This notebook builds that matrix programmatically from the design
*type* of your paper, so it is harder to miss the standard threats. We will:

1. Build a synthetic DiD design where the staggered-adoption pathology is real (so the question
   matters).
2. Enumerate identification threats for *this* design class — parallel trends, anticipation,
   no-other-shocks, SUTVA, staggered-adoption forbidden comparisons, definition-and-measurement,
   selection-into-treatment.
3. For each threat, generate (i) a likely audience question, (ii) the prepared answer, (iii) a
   diagnostic the speaker can show on a backup slide, and (iv) a severity tag.
4. Render the whole thing as a pandas DataFrame you can sort, filter, and export to Markdown for your
   prep packet.

The DGP is synthetic and seeded, no external data — re-running gives the same numbers and the same
diagnostic plots.

In [ ]:
import matplotlib
matplotlib.use("Agg")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyfixest as pf

rng = np.random.default_rng(42)

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["font.size"] = 11

print("numpy   ", np.__version__)
print("pandas  ", pd.__version__)
print("pyfixest", pf.__version__)

## 1. Priya's design: synthetic staggered DiD

The design that motivates the question matrix is Priya's: a staggered roll-out of state climate-
disclosure mandates from 2016 to 2023, with insurer-level outcome `reserves_growth` (percent change in
catastrophic-risk reserves year-over-year). Treatment varies at the *state* level (a state passes the
mandate, all insurers in that state turn on), so clustering is by state. We plant a true ATT of
$+0.6$ percentage points (mandates raise reserves) so we have a known truth to compare against.

The DGP intentionally includes the *pathologies* the question matrix will later ask about:

- **Pre-trend differences** between early and late adopters (early adopters trended *up* in reserves
  even before adoption, because they tended to be states with more hurricane exposure).
- **Anticipation:** insurers in soon-to-adopt states start raising reserves one year before the
  mandate, because they read the legislative tea leaves.
- **Heterogeneous treatment effects** by cohort (the 2018 cohort has a larger ATT than the 2021
  cohort), which is exactly what the staggered-DiD literature has shown can bias two-way fixed-effects
  estimators.
- **Cross-state spillovers** (a SUTVA violation): when a neighboring state adopts, your insurers also
  raise reserves slightly, anticipating contagion of the legislative wave.

Each pathology will reappear as a row in the anticipated-questions matrix. That is the design: we plant
the threats, then enumerate the questions an audience would ask about them.

In [ ]:
# Design parameters
N_STATES   = 25
INSURERS_PER_STATE = 6   # 150 insurers, 8 years -> 1,200 insurer-year rows
YEARS      = list(range(2016, 2024))   # 8 years
TAU_TRUE   = 0.6         # planted true ATT (pp), but heterogeneous across cohorts

# Cohort structure: 6 early (2018), 7 mid (2020), 5 late (2022), 7 never.
cohort = {}
for s in range(N_STATES):
    if   s < 6:   cohort[s] = 2018
    elif s < 13:  cohort[s] = 2020
    elif s < 18:  cohort[s] = 2022
    else:         cohort[s] = np.nan

# Heterogeneous ATT by cohort -- one of the planted threats.
tau_by_cohort = {2018: 1.1, 2020: 0.6, 2022: 0.2}

# Each state has a 'hurricane exposure' that drives BOTH adoption and pre-trends.
hurricane = {s: rng.normal(0.0, 1.0) for s in range(N_STATES)}

# State-level trend differential: hurricane-exposed states were already trending up.
trend_diff = {s: 0.15 * hurricane[s] for s in range(N_STATES)}

# Spillover: pick neighbor pairs (simple ring topology).
neighbors = {s: [(s - 1) % N_STATES, (s + 1) % N_STATES] for s in range(N_STATES)}

rows = []
for s in range(N_STATES):
    G = cohort[s]
    for k in range(INSURERS_PER_STATE):
        insurer_id = s * INSURERS_PER_STATE + k
        alpha_i = hurricane[s] * 0.5 + rng.normal(0, 0.4)   # insurer permanent level
        for t in YEARS:
            # Common year shock.
            lam_t = 0.05 * (t - 2016) + rng.normal(0, 0.2)
            # Cohort-specific pre-trend.
            pre_trend = trend_diff[s] * (t - 2016)
            # Treatment effect (heterogeneous, with one-year anticipation).
            if not np.isnan(G):
                if t >= G:
                    tau = tau_by_cohort[G] * (1 + 0.1 * (t - G))   # builds up slightly
                elif t == G - 1:
                    tau = 0.30   # ONE-YEAR ANTICIPATION
                else:
                    tau = 0.0
            else:
                tau = 0.0
            # Spillover: if either neighbor has adopted, add a small effect.
            neighbor_adopted = any(
                (not np.isnan(cohort[n])) and (t >= cohort[n]) for n in neighbors[s]
            )
            spill = 0.15 if neighbor_adopted else 0.0
            # Outcome
            y = alpha_i + lam_t + pre_trend + tau + spill + rng.normal(0, 0.4)
            rows.append({
                "insurer": insurer_id, "state": s, "year": t,
                "cohort_G": G, "treat": int((not np.isnan(G)) and (t >= G)),
                "hurricane_exposure": hurricane[s], "reserves_growth": y,
            })

panel = pd.DataFrame(rows)
panel["adopter"] = panel["cohort_G"].notna().astype(int)
print(f"insurer-year rows : {len(panel):,}")
print(f"states            : {panel['state'].nunique()}")
print(f"cohort sizes      :")
for G, sz in panel.groupby("cohort_G", dropna=False)["state"].nunique().items():
    label = "never" if pd.isna(G) else int(G)
    print(f"    {label} : {sz} states")
print(f"PLANTED truth     : ATT varies by cohort -- 2018={tau_by_cohort[2018]}, "
      f"2020={tau_by_cohort[2020]}, 2022={tau_by_cohort[2022]}")
panel.head()

## 2. The naive estimate, and a hint of what could go wrong

Run the two-way fixed-effects DiD an audience will see on Priya's headline slide. The coefficient is
a *weighted average* of cohort-specific ATTs, with weights that depend on cohort size and timing — and,
notoriously, can put *negative* weights on some 2x2 comparisons when treatment is staggered and effects
are heterogeneous. The naive estimate is what we will then probe with the anticipated questions.

In [ ]:
naive = pf.feols(
    "reserves_growth ~ treat | insurer + year",
    data=panel, vcov={"CRV1": "state"},
)
beta_naive = float(naive.coef()["treat"])
se_naive   = float(naive.se()["treat"])
print(f"Naive TWFE DiD: beta = {beta_naive:+.3f} pp  (cluster SE = {se_naive:.3f})")
print(f"Cohort-weighted truth (rough avg of cohort ATTs) ~ {np.mean(list(tau_by_cohort.values())):+.3f} pp")
print(f"The naive estimate may differ from this average because of how TWFE weights cohorts.")

## 3. The threat catalog: identification threats for a DiD design

Now we build the matrix. The catalog below enumerates the *standard* threats a DiD/event-study audience
expects you to address. Each one is associated with: a likely audience question (the literal sentence a
referee would say), a one-sentence prepared answer, a diagnostic the speaker can show on a backup slide,
and a severity tag (`critical` = must have a backup slide; `important` = prepared answer; `minor` =
acknowledge in the limits slide). The catalog is *programmatic*: you instantiate it once for your
design type, and the same catalog will work for any DiD paper. Different design types — RDD, IV, event
studies on returns, synthetic control — would carry different catalogs.

In [ ]:
THREAT_CATALOG = [
    {
        "threat_id": "T01",
        "threat": "Parallel pre-trends violated",
        "question": "Your event-study leads look slightly negative pre-treatment. How do you reconcile "
                    "that with the parallel-trends assumption?",
        "prepared_answer": "Show the event-study leads with 95% CIs; report a joint F-test on the "
                           "lead coefficients; if non-zero, show the result is robust to a "
                           "Callaway-Sant'Anna (2021) estimator that does not require strict parallel "
                           "trends across all cohort-time pairs.",
        "diagnostic_slide": "event-study plot with shaded pre-window + F-test p-value in caption",
        "severity": "critical",
    },
    {
        "threat_id": "T02",
        "threat": "Anticipation effects",
        "question": "If insurers anticipated the mandate and started raising reserves before the rule "
                    "took effect, your k=-1 reference period is contaminated. Did you check?",
        "prepared_answer": "Report the k=-1 and k=-2 coefficients separately; if anticipation is "
                           "present, re-define the reference period to k=-3 or earlier and show "
                           "the headline is unchanged.",
        "diagnostic_slide": "event-study with reference window shifted to k=-3..-5",
        "severity": "critical",
    },
    {
        "threat_id": "T03",
        "threat": "No other contemporaneous shock",
        "question": "Was there any other policy or industry event coinciding with adoption years that "
                    "could explain the jump in reserves?",
        "prepared_answer": "Show a balance check: regress the outcome on year dummies among "
                           "never-adopters only -- those control-state trends should be flat. If "
                           "there is a contemporaneous shock, it would show up there.",
        "diagnostic_slide": "year-effects-only regression on never-adopters, with CIs",
        "severity": "important",
    },
    {
        "threat_id": "T04",
        "threat": "SUTVA violation: cross-state spillovers",
        "question": "Could your effect reflect insurers in non-adopting states responding to "
                    "neighbor-state adoptions? That would violate SUTVA.",
        "prepared_answer": "Re-estimate with a 'treated neighbor' indicator; if the spillover is "
                           "absorbing part of the headline, the coefficient on 'own state adopted' "
                           "should rise, not fall, after partialling out the spillover.",
        "diagnostic_slide": "headline coefficient with vs without 'treated neighbor' control",
        "severity": "critical",
    },
    {
        "threat_id": "T05",
        "threat": "Staggered adoption: forbidden comparisons under TWFE",
        "question": "Goodman-Bacon (2021) shows TWFE estimators can put negative weight on 2x2 "
                    "comparisons under staggered adoption with heterogeneous effects. What's your "
                    "weighted-average decomposition?",
        "prepared_answer": "Show the Goodman-Bacon decomposition: list the weight on each 2x2 (early "
                           "vs late, late vs early as control, treated vs never). If the negative-"
                           "weight component is small, the TWFE estimate is fine; if not, re-report "
                           "with Callaway-Sant'Anna or de Chaisemartin-D'Haultfoeuille.",
        "diagnostic_slide": "Goodman-Bacon weights table + CS estimate alongside TWFE",
        "severity": "critical",
    },
    {
        "threat_id": "T06",
        "threat": "Heterogeneous treatment effects by cohort",
        "question": "Does the ATT differ across the 2018, 2020, and 2022 cohorts? If so, the TWFE "
                    "coefficient is a hard-to-interpret weighted average.",
        "prepared_answer": "Estimate cohort-specific ATTs (one coefficient per cohort) and report "
                           "them side-by-side with a forest plot. State the policy implication of "
                           "the cohort spread, not just the average.",
        "diagnostic_slide": "forest plot of cohort-specific ATTs with 95% CIs",
        "severity": "important",
    },
    {
        "threat_id": "T07",
        "threat": "Definition and measurement: outcome construction",
        "question": "How exactly is reserves_growth computed? If it's a YoY percent change, are "
                    "you handling firms that switched accounting standards?",
        "prepared_answer": "State the exact formula in one slide; report the result with and without "
                           "the firms that re-stated; the headline should not depend on a handful of "
                           "re-staters.",
        "diagnostic_slide": "table 1 with and without re-staters; sample size delta",
        "severity": "important",
    },
    {
        "threat_id": "T08",
        "threat": "Selection into treatment: adoption is not random",
        "question": "States that adopted earlier had higher hurricane exposure. Doesn't that mean "
                    "your control group is fundamentally different?",
        "prepared_answer": "Report a balance table on pre-treatment covariates; show the DiD result "
                           "with state-by-time-trend controls; if the conclusion survives, the "
                           "selection threat is not driving the headline.",
        "diagnostic_slide": "balance table + DiD with state-specific linear trends",
        "severity": "important",
    },
    {
        "threat_id": "T09",
        "threat": "Clustered SE: too few clusters",
        "question": "You have 25 states. Cluster-robust SEs with fewer than 30 clusters can be "
                    "biased downward. Did you use wild-cluster bootstrap?",
        "prepared_answer": "Report the wild-cluster bootstrap p-value alongside the conventional "
                           "CR1 SE. If the bootstrap p-value crosses 0.05 when the CR1 doesn't, the "
                           "result is fragile and you say so.",
        "diagnostic_slide": "table with CR1 SE + WCB p-value side by side",
        "severity": "minor",
    },
    {
        "threat_id": "T10",
        "threat": "External validity: insurance only",
        "question": "Does the effect you find for insurance generalize to other industries with "
                    "climate-disclosure mandates?",
        "prepared_answer": "Acknowledge this is a single-industry estimate; explain why insurance "
                           "is a particularly informative test (reserves are forward-looking and "
                           "regulatorily binding); flag the generalization as future work.",
        "diagnostic_slide": "(none -- this is a limits-slide acknowledgement)",
        "severity": "minor",
    },
]

print(f"threat catalog: {len(THREAT_CATALOG)} entries")
sev_counts = pd.Series([t["severity"] for t in THREAT_CATALOG]).value_counts()
print(sev_counts.to_string())

## 4. The matrix as a DataFrame

The catalog above is a list of dicts; a *matrix* is a sortable, filterable, exportable object. We turn
it into a pandas DataFrame, add a revision-cost column (how many hours of work the diagnostic costs),
and order by severity so the speaker prepares the critical threats first.

In [ ]:
matrix = pd.DataFrame(THREAT_CATALOG)

# Add an estimate of how long the diagnostic would take to build (hours).
COST_HRS = {
    "T01": 2.0,  "T02": 1.5,  "T03": 1.0,  "T04": 3.0,  "T05": 4.0,
    "T06": 2.0,  "T07": 1.0,  "T08": 1.5,  "T09": 0.5,  "T10": 0.0,
}
matrix["cost_hrs"] = matrix["threat_id"].map(COST_HRS)

# Severity to sortable rank.
SEV_RANK = {"critical": 0, "important": 1, "minor": 2}
matrix["sev_rank"] = matrix["severity"].map(SEV_RANK)
matrix = matrix.sort_values(["sev_rank", "cost_hrs"], ascending=[True, False]).drop(columns="sev_rank")

print(f"Critical threats : {(matrix['severity'] == 'critical').sum()}")
print(f"Important threats: {(matrix['severity'] == 'important').sum()}")
print(f"Minor threats    : {(matrix['severity'] == 'minor').sum()}")
print(f"Total prep hours : {matrix['cost_hrs'].sum():.1f}")
print()
with pd.option_context("display.max_colwidth", 70, "display.width", 200):
    print(matrix[["threat_id", "threat", "severity", "cost_hrs"]].to_string(index=False))

## 5. Showing the matrix, one row at a time

For the prep packet (a Markdown file you actually read the night before the talk), we want each row
formatted as a *block* — question on top, answer below, diagnostic to bring up if asked. Let us format
the top three critical threats in that style.

In [ ]:
def format_threat_block(row):
    return (
        f"### {row['threat_id']} — {row['threat']}  [{row['severity'].upper()}, {row['cost_hrs']} h]\n"
        f"**Question they will ask:** {row['question']}\n\n"
        f"**Prepared answer:** {row['prepared_answer']}\n\n"
        f"**Backup slide:** {row['diagnostic_slide']}\n"
    )

critical = matrix.query("severity == 'critical'")
print("=" * 88)
for _, row in critical.iterrows():
    print(format_threat_block(row))
    print("-" * 88)

## 6. One diagnostic in action: the event study (for T01 and T02)

The matrix promises a diagnostic for each threat; let us actually *build* the one for T01 (parallel
pre-trends) and T02 (anticipation). It is a hand-built event study with the reference period at $k=-1$,
and we annotate the $k=-1$ and $k=-2$ coefficients explicitly so the speaker can point at them in the
inevitable question. If the speaker is asked about T02 specifically, they will switch to a backup
version with the reference window shifted to $k \leq -3$.

In [ ]:
# Relative event time, with never-adopters parked at -1 (reference).
panel_es = panel.copy()
panel_es["reltime"] = np.where(
    panel_es["adopter"] == 1,
    panel_es["year"] - panel_es["cohort_G"],
    -1,
)
panel_es["reltime"] = panel_es["reltime"].clip(-4, 4).astype(int)

event = pf.feols(
    "reserves_growth ~ i(reltime, ref=-1) | insurer + year",
    data=panel_es, vcov={"CRV1": "state"},
)
es = event.tidy().reset_index().rename(columns={"Coefficient": "term"})
es = es[es["term"].str.startswith("reltime::")].copy()
es["k"] = es["term"].str.replace("reltime::", "", regex=False).astype(int)
es = es.sort_values("k").reset_index(drop=True)
es = es.rename(columns={"Estimate": "beta", "2.5%": "lo", "97.5%": "hi"})
print(es[["k", "beta", "Std. Error", "lo", "hi"]].round(3).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5))
ax.errorbar(
    es["k"], es["beta"],
    yerr=[es["beta"] - es["lo"], es["hi"] - es["beta"]],
    fmt="o", color="black", ecolor="0.35", elinewidth=1.4, capsize=3, markersize=5,
)
ax.axhline(0.0, color="0.5", linewidth=1.0)
ax.axvline(-1.0, color="0.5", linestyle="--", linewidth=1.0)

# Highlight k=-1 (anticipation) and pre-window F-test region.
k_neg1 = es.loc[es["k"] == -1].iloc[0] if (es["k"] == -1).any() else None
if k_neg1 is not None:
    ax.annotate(
        f"k=-1: {k_neg1['beta']:+.2f}\n(anticipation?)",
        xy=(-1, k_neg1["beta"]), xytext=(-3.5, 0.55),
        fontsize=9, color="0.20",
        arrowprops=dict(arrowstyle="->", color="0.4", lw=1),
    )

ax.set_xlabel("Event time  $k = t - G_s$  (years relative to disclosure mandate)")
ax.set_ylabel("Effect on reserves growth (pp)")
ax.set_title("Event study: diagnostic for T01 (parallel trends) and T02 (anticipation)")
ax.set_xticks(sorted(es["k"].tolist()))
fig.tight_layout()
fig.savefig("/tmp/nb92_event_study.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("event-study diagnostic saved to /tmp/nb92_event_study.png")

# Joint test on pre-period leads (k <= -2).
pre_leads = es[es["k"] <= -2]
print(f"\npre-treatment leads (k <= -2):")
print(pre_leads[["k", "beta", "lo", "hi"]].round(3).to_string(index=False))
print(f"  -> if these straddle zero, the parallel-trends story holds; if not, T01 fires.")

## 7. Exporting the matrix to Markdown for the prep packet

The matrix lives in a notebook while you build it; the night before the talk, you want it as a
plain-text Markdown file you can read on a phone in the hotel lobby. The export below produces exactly
that: the matrix as a Markdown table, with one block-formatted page per critical threat appended.

In [ ]:
md_lines = ["# Anticipated-Questions Matrix — Priya, climate-disclosure DiD\n"]

# Summary table.
md_lines.append("## Summary table\n")
table_view = matrix[["threat_id", "threat", "severity", "cost_hrs"]].copy()
md_lines.append(table_view.to_markdown(index=False))
md_lines.append("\n")

# Detail blocks for the critical threats.
md_lines.append("\n## Critical threats — prepare a backup slide for each\n")
for _, row in matrix.query("severity == 'critical'").iterrows():
    md_lines.append(format_threat_block(row))

# Important threats — one paragraph each, no backup slide required.
md_lines.append("\n## Important threats — prepared one-sentence answers\n")
for _, row in matrix.query("severity == 'important'").iterrows():
    md_lines.append(format_threat_block(row))

prep_md = "\n".join(md_lines)
with open("/tmp/nb92_prep_packet.md", "w") as fh:
    fh.write(prep_md)
print(f"prep packet written to /tmp/nb92_prep_packet.md ({len(prep_md)} chars)")
print("\n--- first 600 chars ---")
print(prep_md[:600])

## 8. The matrix is a *living* object

The matrix you take to the symposium is not the matrix you started with. Every time you give the talk
to your advisor, your lab-mate, or yourself in front of a mirror, *they will ask a question you did not
anticipate*. Each such question becomes a new row. Run the matrix-update cell below to add a row
representing the question Priya was asked in her advisor's office last Tuesday: "Have you checked
whether the result is driven by Florida alone?" That kind of question is genuinely unpredictable from
the catalog — it is specific to *this* paper, *this* dataset — and the matrix only gets useful when you
add the paper-specific rows to the design-class rows.

In [ ]:
# Add a paper-specific row.
new_row = pd.DataFrame([{
    "threat_id":       "T11",
    "threat":          "Driven by one outlier state (Florida)",
    "question":        "Florida is half your hurricane-exposed sample. Is the effect driven by Florida?",
    "prepared_answer": "Drop Florida and re-estimate; the headline coefficient should change by at most 20% "
                       "if Florida is not the whole story; report both estimates.",
    "diagnostic_slide": "headline coefficient with and without Florida; sample size delta",
    "severity":        "critical",
    "cost_hrs":        1.0,
}])
matrix = pd.concat([matrix, new_row], ignore_index=True)
matrix["sev_rank"] = matrix["severity"].map(SEV_RANK)
matrix = matrix.sort_values(["sev_rank", "cost_hrs"], ascending=[True, False]).drop(columns="sev_rank")

print(f"matrix now has {len(matrix)} rows; total prep hours = {matrix['cost_hrs'].sum():.1f}")
print("\nupdated matrix (severity-sorted):")
with pd.option_context("display.max_colwidth", 60, "display.width", 200):
    print(matrix[["threat_id", "threat", "severity", "cost_hrs"]].to_string(index=False))

## 9. The matrix as risk allocation: total cost and what to cut

Priya now has eleven threats to prepare. The total prep cost is about 18 hours. The symposium is in ten
days. If she has only six hours, which threats does she prepare and which does she skip? The simple
rule is: cover all *critical* threats first; cover *important* threats in descending cost-effectiveness
(answer-quality per hour); leave *minor* threats for the limits slide.

The code below sorts the matrix by *priority score* — a composite of severity and cost — so Priya can
read off the top six hours of work that maximally cover the question space.

In [ ]:
# Priority score: weight critical=3, important=1, minor=0.2; divide by cost (small-cost items win ties).
SEV_WEIGHT = {"critical": 3.0, "important": 1.0, "minor": 0.2}
matrix["score"] = matrix["severity"].map(SEV_WEIGHT) / (matrix["cost_hrs"].clip(lower=0.5))

priority = matrix.sort_values("score", ascending=False).reset_index(drop=True)
priority["cum_hrs"] = priority["cost_hrs"].cumsum()

print("Prioritized prep schedule (highest score first):")
with pd.option_context("display.max_colwidth", 55, "display.width", 200):
    print(priority[["threat_id", "threat", "severity", "cost_hrs", "score", "cum_hrs"]]
          .round(2).to_string(index=False))

BUDGET_HRS = 6.0
take = priority[priority["cum_hrs"] <= BUDGET_HRS]
print(f"\nWith a {BUDGET_HRS:.0f}-hour budget, prepare:")
for _, row in take.iterrows():
    print(f"  [{row['severity']:9}] {row['threat_id']}  {row['threat']}  ({row['cost_hrs']} h)")
covered_critical = (take["severity"] == "critical").sum()
total_critical = (matrix["severity"] == "critical").sum()
print(f"\nCritical threats covered: {covered_critical} / {total_critical}")

## 10. The point of the matrix

The matrix is not a clever trick. It is a discipline that turns "preparing for questions" from a vague
anxiety into a concrete list of tasks with severity tags and hour estimates. Priya now knows: she has
five critical threats, six important ones, eleven rows total, and a six-hour budget covers the highest-
priority subset. She also knows which diagnostics need to be on backup slides versus answered with one
sentence. None of that was visible before she built the matrix.

Two habits to carry forward:

- **Generate the matrix from the design type.** If your paper is DiD, you inherit roughly the
  ten-threat catalog above. If it is RDD, IV, or synthetic control, the catalog is different but
  similarly fixed. Building it once and reusing it is the leverage move.
- **The matrix is your script for the Q&A.** When the moderator says "questions?" and a hand goes up,
  you should have *already heard* the question in your prep — and your answer is the row from the
  matrix, not an improvisation. That is the difference between a Priya who walks out of the symposium
  with momentum and a Priya who replays a missed answer for a week.

The exact catalog for non-DiD designs — and the symposium-day protocol for when an audience asks a
question that is *not* in your matrix — lives in the chapter prose.

---

### Your Turn

1. **Build the catalog for a different design.** Take Sam's intraday momentum paper (an event study on
   stock returns around the 9:30am open). Write the threat-catalog list-of-dicts that would generate
   *his* matrix. You should get at least eight rows; the threats overlap with the DiD catalog but the
   identification assumptions are different (no anticipation if events are unscheduled; no
   forbidden-comparisons issue because there is no staggered adoption; etc.).
2. **Run T05 for real.** Use `pyfixest` or write a hand-rolled cohort-by-cohort decomposition to
   approximate the Goodman-Bacon weights on Priya's data. Which 2x2 comparisons get negative weight?
   If you re-estimate with only the clean "treated vs never-treated" 2x2s, how different is the
   coefficient? Report a one-paragraph defense Priya could read aloud.
3. **Generate the prep packet for your own paper.** Take whatever paper you are presenting at the
   symposium, instantiate the threat catalog for its design type, and produce the Markdown prep
   packet. Then time yourself: can you read every prepared answer aloud in under 30 seconds?